In [ ]:
# ============================================================
# MONAI DICOM Reader Benchmark
#
# Compare:
#   1. PyDICOM
#   2. SimpleITK
#   3. MONAI PydicomReader
#
# Metrics:
#   • Read success
#   • Metadata consistency
#   • Pixel consistency
#   • Shape
#   • Dtype
#   • Multi-frame support
#   • Performance
#
# Author:
# ============================================================

In [ ]:
import os
import time
import traceback
from pathlib import Path

import numpy as np
import pandas as pd

import pydicom
import SimpleITK as sitk

from monai.data import PydicomReader

In [ ]:
# ============================================================
# ROOTS
# ============================================================
BASE_DIR = Path(".").resolve()

DATASET = BASE_DIR / "Datasets"

WORKSPACE_DIR = BASE_DIR.parent

# ------------------------------------------------------------
# INPUT DATASET FOLDERS
# ------------------------------------------------------------

DATASET_ROOTS = [
    DATASET / "Aliza",
    DATASET / "AIIMS",
    DATASET / "Breast_Lesions_USG",
    DATASET / "MONAI",
    DATASET / "ReMIND",
    DATASET / "Heartcycle",
    DATASET / "UTA4",
    DATASET / "UTA7",
    DATASET / "UTA10",
    DATASET / "TCIA_IDC_Prostate_MRI_US_Biopsy",
    
]
# ============================================================
# OUTPUT
# ============================================================

OUTPUT_DIR = Path("benchmark_results")
OUTPUT_DIR.mkdir(exist_ok=True)

CSV_PATH = OUTPUT_DIR / "benchmark.csv"

In [ ]:
# ============================================================
# DICOM VALIDATION
# ============================================================

def is_dicom_file(path):

    try:

        with open(path, "rb") as f:

            f.seek(128)

            if f.read(4) == b"DICM":
                return True

        ds = pydicom.dcmread(
            path,
            stop_before_pixels=True,
            force=False
        )

        return any([
            hasattr(ds, "PatientID"),
            hasattr(ds, "StudyInstanceUID"),
            hasattr(ds, "SeriesInstanceUID"),
            hasattr(ds, "SOPInstanceUID")
        ])

    except Exception:

        return False

# ============================================================
# DISCOVERY
# ============================================================

def collect_dicom_files(roots):

    files = []

    print("="*70)
    print("Collecting DICOM files")
    print("="*70)

    for root in roots:

        if not root.exists():

            continue

        print(root)

        for f in root.rglob("*"):

            if not f.is_file():
                continue

            if f.suffix.lower() in [".dcm",".dicom"]:

                if is_dicom_file(f):

                    files.append(f)

            elif f.suffix == "":

                if is_dicom_file(f):

                    files.append(f)

    files = sorted(files)

    print()

    print("Total DICOM:",len(files))

    return files

# ============================================================
# BASIC METADATA
# ============================================================

def metadata_from_dataset(ds):

    meta = {}

    tags = [

        "PatientID",

        "Modality",

        "Manufacturer",

        "ManufacturerModelName",

        "Rows",

        "Columns",

        "BitsAllocated",

        "BitsStored",

        "SamplesPerPixel",

        "PhotometricInterpretation",

        "PixelRepresentation",

        "TransferSyntaxUID",

        "NumberOfFrames"

    ]

    for tag in tags:

        try:

            meta[tag] = str(ds.get(tag,""))

        except:

            meta[tag] = ""

    return meta

# ============================================================
# PIXEL STATISTICS
# ============================================================

def pixel_statistics(arr):

    if arr is None:

        return None

    arr = np.asarray(arr)

    stats = {}

    stats["shape"] = arr.shape

    stats["dtype"] = str(arr.dtype)

    stats["min"] = float(arr.min())

    stats["max"] = float(arr.max())

    stats["mean"] = float(arr.mean())

    stats["std"] = float(arr.std())

    return stats

# ============================================================
# ARRAY COMPARISON
# ============================================================

def compare_arrays(a,b):

    result = {}

    if a is None or b is None:

        result["equal"] = False

        result["rmse"] = np.nan

        result["max_error"] = np.nan

        return result

    a = np.asarray(a)

    b = np.asarray(b)

    result["shape_equal"] = a.shape == b.shape

    result["dtype_equal"] = a.dtype == b.dtype

    if a.shape != b.shape:

        result["equal"] = False

        result["rmse"] = np.nan

        result["max_error"] = np.nan

        return result

    result["equal"] = np.array_equal(a,b)

    diff = a.astype(np.float64)-b.astype(np.float64)

    result["rmse"] = np.sqrt(np.mean(diff**2))

    result["max_error"] = np.max(np.abs(diff))

    return result

# ============================================================
# TIMER
# ============================================================

class Timer:

    def __enter__(self):

        self.t0 = time.perf_counter()

        return self

    def __exit__(self,*args):

        self.elapsed = time.perf_counter()-self.t0

In [2]:
# ============================================================
# READER : PYDICOM
# ============================================================

def read_with_pydicom(dcm_path):

    result = {
        "success": False,
        "reader": "PyDICOM",
        "time": np.nan,
        "exception": "",
        "dataset": None,
        "pixels": None,
        "metadata": {},
        "stats": {},
    }

    try:

        with Timer() as t:

            ds = pydicom.dcmread(
                str(dcm_path),
                force=True
            )

            pixels = ds.pixel_array

        result["success"] = True
        result["time"] = t.elapsed
        result["dataset"] = ds
        result["pixels"] = pixels
        result["metadata"] = metadata_from_dataset(ds)
        result["stats"] = pixel_statistics(pixels)

    except Exception as e:

        result["exception"] = traceback.format_exc()

    return result


# ============================================================
# READER : SimpleITK
# ============================================================

def read_with_simpleitk(dcm_path):

    result = {
        "success": False,
        "reader": "SimpleITK",
        "time": np.nan,
        "exception": "",
        "pixels": None,
        "stats": {},
        "spacing": None,
        "origin": None,
        "direction": None,
    }

    try:

        with Timer() as t:

            image = sitk.ReadImage(str(dcm_path))

            pixels = sitk.GetArrayFromImage(image)

        result["success"] = True
        result["time"] = t.elapsed
        result["pixels"] = pixels
        result["stats"] = pixel_statistics(pixels)

        try:
            result["spacing"] = image.GetSpacing()
        except:
            pass

        try:
            result["origin"] = image.GetOrigin()
        except:
            pass

        try:
            result["direction"] = image.GetDirection()
        except:
            pass

    except Exception:

        result["exception"] = traceback.format_exc()

    return result


# ============================================================
# INITIALIZE MONAI READER
# ============================================================

try:

    monai_reader = PydicomReader()

except Exception:

    monai_reader = None


# ============================================================
# READER : MONAI
# ============================================================

def read_with_monai(dcm_path):

    result = {
        "success": False,
        "reader": "MONAI",
        "time": np.nan,
        "exception": "",
        "pixels": None,
        "stats": {},
        "metadata": {},
    }

    if monai_reader is None:

        result["exception"] = "MONAI PydicomReader unavailable"

        return result

    try:

        with Timer() as t:

            img = monai_reader.read(str(dcm_path))

            pixels, meta = monai_reader.get_data(img)

        result["success"] = True
        result["time"] = t.elapsed

        result["pixels"] = np.asarray(pixels)

        result["stats"] = pixel_statistics(result["pixels"])

        if isinstance(meta, dict):

            result["metadata"] = meta

    except Exception:

        result["exception"] = traceback.format_exc()

    return result


# ============================================================
# PRINT READER SUMMARY
# ============================================================

def print_reader_summary(result):

    print("=" * 70)

    print(result["reader"])

    print("=" * 70)

    print("Success :", result["success"])

    print("Time    :", result["time"])

    if result["success"]:

        stats = result["stats"]

        print("Shape   :", stats["shape"])
        print("Dtype   :", stats["dtype"])
        print("Min     :", stats["min"])
        print("Max     :", stats["max"])
        print("Mean    :", stats["mean"])
        print("Std     :", stats["std"])

    else:

        print(result["exception"])


# ============================================================
# QUICK SANITY TEST
# ============================================================

candidate_files = collect_dicom_files(DATASET_ROOTS)

print()

print("Found", len(candidate_files), "DICOM files")

if len(candidate_files):

    test_file = candidate_files[0]

    print("\nTesting readers on:\n")
    print(test_file)

    py = read_with_pydicom(test_file)

    sitk_result = read_with_simpleitk(test_file)

    monai = read_with_monai(test_file)

    print_reader_summary(py)

    print_reader_summary(sitk_result)

    print_reader_summary(monai)

C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\Aliza
C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AIIMS
C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\Breast_Lesions_USG
C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\MONAI
C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\ReMIND
C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\Heartcycle
C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UTA4
C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UTA7
C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UTA10
C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\TCIA_IDC_Prostate_MRI_US_Biopsy

Total DICOM: 3468

Found 3468 DICOM files

Testing readers on:

C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AIIMS\AIIMSJ_GE_Vivid_E95\Q3EDC380.dcm
PyDICOM
Success : True
Time    : 1.308778499999999
Shape   : (126, 708, 1016, 3)
Dtype   : uint8
Min     : 0.0
Max     : 255.0
Mean    : 92.70084474124704
Std     : 54.46496664853589
SimpleITK
Success : True
Time    : 2.0651714000000005
Shape   : (126, 708, 1016, 3)
Dtype   : uint8
Min     : 0.0
Max     : 25

In [3]:
# ============================================================
# DICOM METADATA EXTRACTION
# ============================================================

def extract_dicom_information(ds):

    info = {}

    tags = [

        "SOPClassUID",

        "TransferSyntaxUID",

        "PhotometricInterpretation",

        "SamplesPerPixel",

        "BitsAllocated",

        "BitsStored",

        "HighBit",

        "PixelRepresentation",

        "Rows",

        "Columns",

        "PlanarConfiguration",

        "NumberOfFrames",

        "Manufacturer",

        "ManufacturerModelName",

        "Modality",

        "UltrasoundColorDataPresent",

        "LossyImageCompression",

        "LossyImageCompressionMethod"

    ]

    for tag in tags:

        try:

            info[tag] = str(ds.get(tag, ""))

        except:

            info[tag] = ""

    return info

In [4]:
# ============================================================
# PHOTOMETRIC
# ============================================================

def validate_photometric(ds):

    report = {}

    report["Photometric"] = str(
        ds.get(
            "PhotometricInterpretation",
            ""
        )
    )

    report["SamplesPerPixel"] = int(
        ds.get(
            "SamplesPerPixel",
            1
        )
    )

    report["PlanarConfiguration"] = str(
        ds.get(
            "PlanarConfiguration",
            ""
        )
    )

    report["Color"] = \
        report["SamplesPerPixel"] > 1

    return report

In [5]:
# ============================================================
# TRANSFER SYNTAX
# ============================================================

def classify_transfer_syntax(ds):

    uid = str(
        ds.file_meta.TransferSyntaxUID
    )

    if uid == "1.2.840.10008.1.2":

        return "Implicit VR Little"

    if uid == "1.2.840.10008.1.2.1":

        return "Explicit VR Little"

    if uid == "1.2.840.10008.1.2.2":

        return "Explicit VR Big"

    if uid.startswith("1.2.840.10008.1.2.4"):

        return "JPEG"

    if uid.startswith("1.2.840.10008.1.2.5"):

        return "RLE"

    return uid

In [6]:
# ============================================================
# FAILURE CLASSIFICATION
# ============================================================

def classify_failure(py,
                     sitk_result,
                     monai):

    report = {}

    report["Status"] = "PASS"

    report["Reason"] = ""

    if not py["success"]:

        report["Status"] = "FAIL"

        report["Reason"] = "PyDICOM Read Failure"

        return report

    if not sitk_result["success"]:

        report["Status"] = "FAIL"

        report["Reason"] = "SimpleITK Read Failure"

        return report

    if not monai["success"]:

        report["Status"] = "FAIL"

        report["Reason"] = "MONAI Read Failure"

        return report

    p = canonicalize_array(py["pixels"])

    s = canonicalize_array(sitk_result["pixels"])

    m = canonicalize_array(monai["pixels"])

    if p.shape != s.shape:

        report["Status"] = "WARNING"

        report["Reason"] = \
            "PyDICOM vs SimpleITK Shape"

    elif p.shape != m.shape:

        report["Status"] = "WARNING"

        report["Reason"] = \
            "PyDICOM vs MONAI Shape"

    elif not np.array_equal(p,s):

        report["Status"] = "WARNING"

        report["Reason"] = \
            "Pixel Difference PyDICOM/SimpleITK"

    elif not np.array_equal(p,m):

        report["Status"] = "WARNING"

        report["Reason"] = \
            "Pixel Difference PyDICOM/MONAI"

    return report

In [7]:
# ============================================================
# CANONICALIZE ARRAY
#
# Convert every reader output to the same convention
#
# Final convention:
#
#   Grayscale
#       (H,W)
#       (Frames,H,W)
#
#   RGB
#       (H,W,3)
#       (Frames,H,W,3)
# ============================================================

def canonicalize_array(arr):

    if arr is None:
        return None

    arr = np.asarray(arr)

    # -------------------------
    # Already 2D
    # -------------------------

    if arr.ndim == 2:
        return arr

    # -------------------------
    # RGB image
    # -------------------------

    if arr.ndim == 3:

        if arr.shape[-1] in (3,4):
            return arr

        if arr.shape[0] in (3,4):

            # Channel-first
            return np.moveaxis(arr,0,-1)

        return arr

    # -------------------------
    # Multi-frame RGB
    # -------------------------

    if arr.ndim == 4:

        # (F,C,H,W)

        if arr.shape[1] in (3,4):

            arr = np.moveaxis(arr,1,-1)

            return arr

        return arr

    return arr


# ============================================================
# ARRAY REPORT
# ============================================================

def describe_array(arr):

    if arr is None:

        return "None"

    arr=np.asarray(arr)

    return {

        "shape":arr.shape,

        "dtype":str(arr.dtype),

        "min":float(arr.min()),

        "max":float(arr.max()),

        "mean":float(arr.mean()),

        "std":float(arr.std()),

        "ndim":arr.ndim

    }


# ============================================================
# PIXEL DIFFERENCE REPORT
# ============================================================

def pixel_difference_report(a,b):

    report={}

    a=canonicalize_array(a)

    b=canonicalize_array(b)

    if a is None or b is None:

        report["status"]="Missing"

        return report

    report["shapeA"]=a.shape
    report["shapeB"]=b.shape

    report["dtypeA"]=str(a.dtype)
    report["dtypeB"]=str(b.dtype)

    report["shape_equal"]=a.shape==b.shape

    if a.shape!=b.shape:

        report["status"]="Different Shape"

        return report

    diff=a.astype(np.float64)-b.astype(np.float64)

    report["equal"]=np.array_equal(a,b)

    report["allclose"]=np.allclose(a,b)

    report["rmse"]=np.sqrt(np.mean(diff**2))

    report["mae"]=np.mean(np.abs(diff))

    report["max_error"]=np.max(np.abs(diff))

    report["different_pixels"]=np.count_nonzero(diff)

    report["status"]="Compared"

    return report

In [8]:
# ============================================================
# FRAME COUNT
# ============================================================

def frame_count(arr):

    if arr is None:
        return None

    arr = np.asarray(arr)

    if arr.ndim == 2:
        return 1

    if arr.ndim == 3:

        if arr.shape[-1] in (3,4):
            return 1

        return arr.shape[0]

    if arr.ndim == 4:
        return arr.shape[0]

    return None

In [9]:
def throughput(arr,time_taken):

    if arr is None:

        return np.nan

    pixels = np.asarray(arr).size

    return pixels / time_taken

In [10]:
# ============================================================
# METADATA COMPARISON
# ============================================================

def compare_metadata(meta1, meta2):

    result = {}

    keys = sorted(set(meta1.keys()).union(meta2.keys()))

    for k in keys:

        v1 = str(meta1.get(k, ""))

        v2 = str(meta2.get(k, ""))

        result[k + "_equal"] = (v1 == v2)

    return result


# ============================================================
# BUILD ONE RESULT ROW
# ============================================================

def build_result_row(dcm_file,
                     py,
                     sitk_result,
                     monai):

    row = {}

    # --------------------------------------------------------
    # File information
    # --------------------------------------------------------

    row["File"] = str(dcm_file)

    row["Filename"] = dcm_file.name

    row["ReaderCount"] = \
        int(py["success"]) + \
        int(sitk_result["success"]) + \
        int(monai["success"])

    # --------------------------------------------------------
    # Success
    # --------------------------------------------------------

    row["PyDICOM_OK"] = py["success"]

    row["SimpleITK_OK"] = sitk_result["success"]

    row["MONAI_OK"] = monai["success"]

    # --------------------------------------------------------
    # Read Time
    # --------------------------------------------------------

    row["Py_Time"] = py["time"]

    row["SimpleITK_Time"] = sitk_result["time"]

    row["MONAI_Time"] = monai["time"]

    # --------------------------------------------------------
    # Throughput
    # --------------------------------------------------------
    
    row["Py_Throughput"] = throughput(
        py["pixels"],
        py["time"]
    )
    
    row["SimpleITK_Throughput"] = throughput(
        sitk_result["pixels"],
        sitk_result["time"]
    )
    
    row["MONAI_Throughput"] = throughput(
        monai["pixels"],
        monai["time"]
    )

    # --------------------------------------------------------
    # Shape
    # --------------------------------------------------------

    if py["success"]:
        row["Py_Shape"] = str(py["stats"]["shape"])
        row["Py_Dtype"] = py["stats"]["dtype"]
    else:
        row["Py_Shape"] = ""
        row["Py_Dtype"] = ""

    if sitk_result["success"]:
        row["SimpleITK_Shape"] = str(
            sitk_result["stats"]["shape"]
        )
        row["SimpleITK_Dtype"] = \
            sitk_result["stats"]["dtype"]
    else:
        row["SimpleITK_Shape"] = ""
        row["SimpleITK_Dtype"] = ""

    if monai["success"]:
        row["MONAI_Shape"] = \
            str(monai["stats"]["shape"])
        row["MONAI_Dtype"] = \
            monai["stats"]["dtype"]
    else:
        row["MONAI_Shape"] = ""
        row["MONAI_Dtype"] = ""

    # --------------------------------------------------------
    # Pixel comparisons
    # --------------------------------------------------------

    ps = compare_arrays(
        py["pixels"],
        sitk_result["pixels"]
    )

    pm = compare_arrays(
        py["pixels"],
        monai["pixels"]
    )

    sm = compare_arrays(
        sitk_result["pixels"],
        monai["pixels"]
    )

    # ------------------------

    row["Py_vs_SimpleITK_Equal"] = \
        ps["equal"]

    row["Py_vs_SimpleITK_RMSE"] = \
        ps["rmse"]

    row["Py_vs_SimpleITK_MaxError"] = \
        ps["max_error"]

    # ------------------------

    row["Py_vs_MONAI_Equal"] = \
        pm["equal"]

    row["Py_vs_MONAI_RMSE"] = \
        pm["rmse"]

    row["Py_vs_MONAI_MaxError"] = \
        pm["max_error"]

    # ------------------------

    row["SimpleITK_vs_MONAI_Equal"] = \
        sm["equal"]

    row["SimpleITK_vs_MONAI_RMSE"] = \
        sm["rmse"]

    row["SimpleITK_vs_MONAI_MaxError"] = \
        sm["max_error"]

    # --------------------------------------------------------
    # Metadata
    # --------------------------------------------------------

    if py["success"]:

        row.update(py["metadata"])

    return row


# ============================================================
# BENCHMARK
# ============================================================

results = []

print()

print("="*70)
print("Running Benchmark")
print("="*70)

total = len(candidate_files)

for idx, dcm_file in enumerate(candidate_files):

    print()

    print(f"[{idx+1}/{total}] {dcm_file.name}")

    py = read_with_pydicom(dcm_file)

    sitk_result = read_with_simpleitk(dcm_file)
    
    monai = read_with_monai(dcm_file)
    
    # -------------------------
    # DICOM metadata
    # -------------------------
    
    ds = py["dataset"]
    
    dicom_info = extract_dicom_information(ds)
    
    photo_info = validate_photometric(ds)
    
    syntax = classify_transfer_syntax(ds)
    
    failure = classify_failure(
        py,
        sitk_result,
        monai
    )
    
    # -------------------------
    # Build result row
    # -------------------------
    
    row = build_result_row(
        dcm_file,
        py,
        sitk_result,
        monai
    )
    
    row.update(dicom_info)
    
    row.update(photo_info)
    
    row["TransferSyntax"] = syntax
    
    row["ValidationStatus"] = failure["Status"]
    
    row["FailureReason"] = failure["Reason"]
    
    row["Frames_PyDICOM"] = frame_count(py["pixels"])
    
    row["Frames_SimpleITK"] = frame_count(
        sitk_result["pixels"]
    )
    
    row["Frames_MONAI"] = frame_count(
        monai["pixels"]
    )
    
    results.append(row)

print()

print("="*70)
print("Benchmark Finished")
print("="*70)


Running Benchmark

[1/3468] Q3EDC380.dcm

[2/3468] Q3EDCA02.dcm

[3/3468] Q3EDCBG4.dcm

[4/3468] Q3EDCH86.dcm

[5/3468] Q3EDCK08.dcm

[6/3468] IM_0001.dcm

[7/3468] IM_0002.dcm

[8/3468] usvolume_4D.dcm

[9/3468] US-GE-4AICL142.dcm

[10/3468] US.1.3.46.670589.14.3000.100.2.199999.20110826084400.0.dcm

[11/3468] US.1.3.46.670589.14.3000.100.2.199999.20110826084428.0.dcm

[12/3468] US.1.3.46.670589.14.3000.100.2.199999.20110826084457.0.dcm

[13/3468] US.1.3.46.670589.14.3000.100.2.199999.20110826084523.0.dcm

[14/3468] US.1.3.46.670589.14.3000.100.2.199999.20110826084559.0.dcm

[15/3468] US.1.3.46.670589.14.3000.100.2.199999.20110826084627.0.dcm

[16/3468] US.1.3.46.670589.14.3000.100.2.199999.20110826084654.0.dcm

[17/3468] US.1.3.46.670589.14.3000.100.2.199999.20110826084725.0.dcm

[18/3468] US.1.3.46.670589.14.3000.100.2.199999.20110826084753.0.dcm

[19/3468] US.1.3.46.670589.14.3000.100.2.199999.20110826084825.0.dcm

[20/3468] US.1.3.46.670589.14.3000.100.2.199999.20110826084853.0.d

The decoded RLE segment contains non-conformant padding - 276025 vs. 276024 bytes expected



[30/3468] 8c780494-786d-4ed7-b2a0-24fe18bd8dce.dcm

[31/3468] bf1b0a44-60ca-4d20-a6e2-dde3ae34ea53.dcm

[32/3468] 91d91920-3d47-4a81-8a45-3fa9fdb2b70a.dcm

[33/3468] 99d9bce9-a4d5-4c53-bdb3-65cbd64b0d80.dcm

[34/3468] 0d3220cf-c1e2-45dd-b3e2-af125beb8a15.dcm

[35/3468] 39ea0987-7be5-4b9c-aa5c-a538ab0f085a.dcm

[36/3468] 495f767d-377c-41ed-8df7-c79c44b23d02.dcm

[37/3468] 4a2c814f-ecaa-4f8e-9560-d12d73136891.dcm

[38/3468] 84a0a112-63c6-4aff-85e3-b36715c38f03.dcm

[39/3468] ab591baa-8f19-46e2-87ed-8f65adbcdd28.dcm

[40/3468] 9c0b66c1-7cfc-4532-8465-2653574096bd.dcm

[41/3468] 574a4303-0c53-4982-b2b2-19f0670c94ff.dcm

[42/3468] a8f6217f-e04d-4b2a-92ba-dd13274260af.dcm

[43/3468] cc79a9d0-2a38-4de6-b3b5-a15c97e81143.dcm

[44/3468] 08453398-3c91-4159-adf9-77bd4617c61f.dcm

[45/3468] b6643095-f59a-4981-b8c2-b4b5f6f75977.dcm

[46/3468] 645085d6-7b68-4c47-b8f3-e3cc43506581.dcm

[47/3468] 768d05c1-d6a0-43b9-9902-60ceabd25411.dcm

[48/3468] 4a6fad44-7808-4948-9c61-790a372e0904.dcm

[49/3468] b

In [11]:
# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame(results)

print()

print(df.head())

print()

print("Rows :", len(df))

print("Columns :", len(df.columns))


                                                File      Filename  \
0  C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AI...  Q3EDC380.dcm   
1  C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AI...  Q3EDCA02.dcm   
2  C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AI...  Q3EDCBG4.dcm   
3  C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AI...  Q3EDCH86.dcm   
4  C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AI...  Q3EDCK08.dcm   

   ReaderCount  PyDICOM_OK  SimpleITK_OK  MONAI_OK   Py_Time  SimpleITK_Time  \
0            3        True          True      True  1.278399        2.012048   
1            3        True          True      True  1.245472        2.038860   
2            3        True          True      True  2.416232        3.716097   
3            3        True          True      True  0.030084        0.252722   
4            3        True          True      True  0.043748        0.348895   

   MONAI_Time  Py_Throughput  ...  LossyImageCompression  \
0    1.437810   2.126926e+08  ...    

In [12]:
# ============================================================
# SAVE CSV
# ============================================================

df.to_csv(
    CSV_PATH,
    index=False
)

print()

print("CSV Saved")

print(CSV_PATH)


CSV Saved
benchmark_results\benchmark.csv


# IMPROVED COMPARISON

In [13]:
# ============================================================
# IMPROVED COMPARISON
# ============================================================

improved_results=[]

print("="*80)
print("Detailed Comparison")
print("="*80)

for i,row in enumerate(results):

    dcm=Path(row["File"])

    print(f"{i+1}/{len(results)}")

    py=read_with_pydicom(dcm)

    sitk_result=read_with_simpleitk(dcm)

    monai=read_with_monai(dcm)

    py_pixels=canonicalize_array(py["pixels"])

    sitk_pixels=canonicalize_array(sitk_result["pixels"])

    monai_pixels=canonicalize_array(monai["pixels"])

    py_vs_sitk=pixel_difference_report(
        py_pixels,
        sitk_pixels
    )

    py_vs_monai=pixel_difference_report(
        py_pixels,
        monai_pixels
    )

    sitk_vs_monai=pixel_difference_report(
        sitk_pixels,
        monai_pixels
    )

    improved_results.append({

        "File":str(dcm),

        "Py==SimpleITK":
            py_vs_sitk.get("equal"),

        "Py==MONAI":
            py_vs_monai.get("equal"),

        "SimpleITK==MONAI":
            sitk_vs_monai.get("equal"),

        "Py_MONAI_RMSE":
            py_vs_monai.get("rmse"),

        "Py_SITK_RMSE":
            py_vs_sitk.get("rmse"),

        "SITK_MONAI_RMSE":
            sitk_vs_monai.get("rmse"),

        "Py_MONAI_MaxError":
            py_vs_monai.get("max_error"),

        "Py_SITK_MaxError":
            py_vs_sitk.get("max_error"),

        "SITK_MONAI_MaxError":
            sitk_vs_monai.get("max_error"),

        "Py_MONAI_DifferentPixels":
            py_vs_monai.get("different_pixels"),

        "Py_SITK_DifferentPixels":
            py_vs_sitk.get("different_pixels"),

        "SITK_MONAI_DifferentPixels":
            sitk_vs_monai.get("different_pixels"),

    })

print()

print("Done.")

Detailed Comparison
1/3468
2/3468
3/3468
4/3468
5/3468
6/3468
7/3468
8/3468
9/3468
10/3468
11/3468
12/3468
13/3468
14/3468
15/3468
16/3468
17/3468
18/3468
19/3468
20/3468
21/3468
22/3468
23/3468
24/3468
25/3468
26/3468
27/3468
28/3468
29/3468


The decoded RLE segment contains non-conformant padding - 276025 vs. 276024 bytes expected


30/3468
31/3468
32/3468
33/3468
34/3468
35/3468
36/3468
37/3468
38/3468
39/3468
40/3468
41/3468
42/3468
43/3468
44/3468
45/3468
46/3468
47/3468
48/3468
49/3468
50/3468
51/3468
52/3468
53/3468
54/3468
55/3468
56/3468
57/3468
58/3468
59/3468
60/3468
61/3468
62/3468
63/3468
64/3468
65/3468
66/3468
67/3468
68/3468
69/3468
70/3468
71/3468
72/3468
73/3468
74/3468
75/3468
76/3468
77/3468
78/3468
79/3468
80/3468
81/3468
82/3468
83/3468
84/3468
85/3468
86/3468
87/3468
88/3468
89/3468
90/3468
91/3468
92/3468
93/3468
94/3468
95/3468
96/3468
97/3468
98/3468
99/3468
100/3468
101/3468
102/3468
103/3468
104/3468
105/3468
106/3468
107/3468
108/3468
109/3468
110/3468
111/3468
112/3468
113/3468
114/3468
115/3468
116/3468
117/3468
118/3468
119/3468
120/3468
121/3468
122/3468
123/3468
124/3468
125/3468
126/3468
127/3468
128/3468
129/3468
130/3468
131/3468
132/3468
133/3468
134/3468
135/3468
136/3468
137/3468
138/3468
139/3468
140/3468
141/3468
142/3468
143/3468
144/3468
145/3468
146/3468
147/3468
148/3468

In [14]:
comparison_df=pd.DataFrame(improved_results)

comparison_csv=OUTPUT_DIR/"pixel_comparison.csv"

comparison_df.to_csv(
    comparison_csv,
    index=False
)

comparison_df.head()

,File,Py==SimpleITK,Py==MONAI,SimpleITK==MONAI,Py_MONAI_RMSE,Py_SITK_RMSE,SITK_MONAI_RMSE,Py_MONAI_MaxError,Py_SITK_MaxError,SITK_MONAI_MaxError,Py_MONAI_DifferentPixels,Py_SITK_DifferentPixels,SITK_MONAI_DifferentPixels
0,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AI...,False,None,None,NaN,91.792781,None,NaN,238.0,None,NaN,207166236.0,None
1,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AI...,False,None,None,NaN,91.792778,None,NaN,238.0,None,NaN,207165985.0,None
2,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AI...,False,None,None,NaN,86.959729,None,NaN,238.0,None,NaN,384563199.0,None
3,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AI...,None,None,None,NaN,NaN,None,NaN,NaN,None,NaN,NaN,None
4,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\AI...,None,None,None,NaN,NaN,None,NaN,NaN,None,NaN,NaN,None


In [15]:
print("="*80)
print("SUMMARY")
print("="*80)

pairs=[

    "Py==SimpleITK",

    "Py==MONAI",

    "SimpleITK==MONAI"

]

for p in pairs:

    total=comparison_df[p].count()

    equal=comparison_df[p].sum()

    print()

    print(p)

    print("Equal :",equal)

    print("Different :",total-equal)

    print("Agreement : %.2f%%"%(100*equal/total))

SUMMARY

Py==SimpleITK
Equal : 2083
Different : 9
Agreement : 99.57%

Py==MONAI
Equal : 0
Different : 1296
Agreement : 0.00%

SimpleITK==MONAI
Equal : 0
Different : 0
Agreement : nan%


invalid value encountered in scalar divide


In [16]:
print("="*80)
print("Largest RMSE")
print("="*80)

comparison_df.sort_values(

    "Py_MONAI_RMSE",

    ascending=False

).head(20)

Largest RMSE


,File,Py==SimpleITK,Py==MONAI,SimpleITK==MONAI,Py_MONAI_RMSE,Py_SITK_RMSE,SITK_MONAI_RMSE,Py_MONAI_MaxError,Py_SITK_MaxError,SITK_MONAI_MaxError,Py_MONAI_DifferentPixels,Py_SITK_DifferentPixels,SITK_MONAI_DifferentPixels
3026,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,109.955579,NaN,None,575.0,NaN,None,76192.0,NaN,None
2282,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,109.955579,NaN,None,575.0,NaN,None,76192.0,NaN,None
2525,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,97.008382,NaN,None,737.0,NaN,None,76374.0,NaN,None
3269,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,97.008382,NaN,None,737.0,NaN,None,76374.0,NaN,None
2294,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,96.162756,NaN,None,442.0,NaN,None,75798.0,NaN,None
3038,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,96.162756,NaN,None,442.0,NaN,None,75798.0,NaN,None
2524,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,85.564555,NaN,None,504.0,NaN,None,73616.0,NaN,None
3268,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,85.564555,NaN,None,504.0,NaN,None,73616.0,NaN,None
3267,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,85.476373,NaN,None,518.0,NaN,None,73428.0,NaN,None
2523,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,85.476373,NaN,None,518.0,NaN,None,73428.0,NaN,None


In [17]:
comparison_df.sort_values(

    "Py_MONAI_DifferentPixels",

    ascending=False

).head(20)

,File,Py==SimpleITK,Py==MONAI,SimpleITK==MONAI,Py_MONAI_RMSE,Py_SITK_RMSE,SITK_MONAI_RMSE,Py_MONAI_MaxError,Py_SITK_MaxError,SITK_MONAI_MaxError,Py_MONAI_DifferentPixels,Py_SITK_DifferentPixels,SITK_MONAI_DifferentPixels
3316,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,65.127237,NaN,None,490.0,NaN,None,115388.0,NaN,None
3324,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,62.201784,NaN,None,407.0,NaN,None,115242.0,NaN,None
3315,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,66.566390,NaN,None,495.0,NaN,None,115204.0,NaN,None
2776,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,50.618486,NaN,None,458.0,NaN,None,115198.0,NaN,None
2210,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,50.618486,NaN,None,458.0,NaN,None,115198.0,NaN,None
3318,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,63.966947,NaN,None,445.0,NaN,None,115170.0,NaN,None
3320,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,63.562045,NaN,None,417.0,NaN,None,115148.0,NaN,None
3319,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,63.703809,NaN,None,416.0,NaN,None,115130.0,NaN,None
3462,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,58.563663,NaN,None,667.0,NaN,None,115128.0,NaN,None
2220,C:\Users\Gajendra\Desktop\D3_MONAI\Datasets\UT...,None,False,None,50.053505,NaN,None,487.0,NaN,None,115076.0,NaN,None


In [18]:
print("="*80)
print("Average Read Time")
print("="*80)

print(

    df[

        [

            "Py_Time",

            "SimpleITK_Time",

            "MONAI_Time"

        ]

    ].mean()

)

Average Read Time
Py_Time           0.038962
SimpleITK_Time    0.095818
MONAI_Time        0.120159
dtype: float64


In [19]:
# row.update(dicom_info)

# row.update(photo_info)

# row["TransferSyntax"] = syntax

# row["ValidationStatus"] = failure["Status"]

# row["FailureReason"] = failure["Reason"]

# row["Frames_PyDICOM"] = frame_count(py["pixels"])

# row["Frames_SimpleITK"] = frame_count(sitk_result["pixels"])

# row["Frames_MONAI"] = frame_count(monai["pixels"])

In [20]:
print("="*80)
print("Transfer Syntax Summary")
print("="*80)

print(

    df.groupby("TransferSyntax")

      .size()

      .sort_values(ascending=False)

)

Transfer Syntax Summary
TransferSyntax
Explicit VR Little    3442
RLE                     14
JPEG                    12
dtype: int64


In [21]:
print("="*80)
print("Photometric Interpretation")
print("="*80)

print(

    df.groupby("PhotometricInterpretation")

      .size()

      .sort_values(ascending=False)

)

Photometric Interpretation
PhotometricInterpretation
MONOCHROME2      3425
RGB                17
PALETTE COLOR      14
YBR_FULL_422       11
YBR_FULL            1
dtype: int64


In [22]:
print("="*80)
print("Failure Summary")
print("="*80)

print(

    df.groupby("FailureReason")

      .size()

      .sort_values(ascending=False)

)

Failure Summary
FailureReason
PyDICOM vs MONAI Shape        2092
PyDICOM vs SimpleITK Shape    1376
dtype: int64


In [23]:
print("="*80)
print("Average Read Time")
print("="*80)

print(

    df[

        [

            "Py_Time",

            "SimpleITK_Time",

            "MONAI_Time"

        ]

    ].describe()

)

Average Read Time
           Py_Time  SimpleITK_Time   MONAI_Time
count  3468.000000     3468.000000  3468.000000
mean      0.038962        0.095818     0.120159
std       0.077998        0.141117     0.181800
min       0.001518        0.004220     0.002586
25%       0.002260        0.004999     0.002943
50%       0.032432        0.089417     0.095441
75%       0.057104        0.157111     0.170346
max       2.416232        3.716097     4.079869


In [24]:
# ============================================================
# MISMATCH OUTPUT
# ============================================================

MISMATCH_DIR = OUTPUT_DIR / "Mismatches"
MISMATCH_DIR.mkdir(exist_ok=True)

In [25]:
def get_display_frame(arr):
    """
    Convert any DICOM pixel array to a displayable image.

    Supported inputs:
        (H,W)
        (H,W,3)
        (F,H,W)
        (F,H,W,3)
        (C,H,W)
        (F,C,H,W)

    Returns
        (H,W) or (H,W,3)
    """

    if arr is None:
        return None

    arr = canonicalize_array(arr)

    if arr.ndim == 2:
        return arr

    if arr.ndim == 3:

        # RGB
        if arr.shape[-1] in (3,4):
            return arr

        # Multi-frame grayscale
        return arr[0]

    if arr.ndim == 4:

        # First RGB frame
        return arr[0]

    return arr

In [26]:
import matplotlib.pyplot as plt

def save_comparison_images(
    filename,
    py_img,
    sitk_img,
    monai_img,
):

    # py_img = canonicalize_array(py_img)
    # sitk_img = canonicalize_array(sitk_img)
    # monai_img = canonicalize_array(monai_img)

    py_img = get_display_frame(py_img)
    sitk_img = get_display_frame(sitk_img)
    monai_img = get_display_frame(monai_img)

    if py_img is None or sitk_img is None or monai_img is None:
        return

    # first frame only

    if py_img.ndim == 3 and py_img.shape[-1] not in (3,4):
        py_img = py_img[0]

    if sitk_img.ndim == 3 and sitk_img.shape[-1] not in (3,4):
        sitk_img = sitk_img[0]

    if monai_img.ndim == 3 and monai_img.shape[-1] not in (3,4):
        monai_img = monai_img[0]

    fig,ax = plt.subplots(1,3,figsize=(15,5))

    ax[0].imshow(py_img,cmap="gray")
    ax[0].set_title("PyDICOM")

    ax[1].imshow(sitk_img,cmap="gray")
    ax[1].set_title("SimpleITK")

    ax[2].imshow(monai_img,cmap="gray")
    ax[2].set_title("MONAI")

    for a in ax:
        a.axis("off")

    plt.tight_layout()

    plt.savefig(
        MISMATCH_DIR / (filename + ".png"),
        dpi=200
    )

    plt.close()

In [27]:
def save_difference_image(
    filename,
    img1,
    img2,
    title,
):

    img1 = canonicalize_array(img1)
    img2 = canonicalize_array(img2)

    if img1.shape != img2.shape:
        return

    if img1.ndim == 3 and img1.shape[-1] not in (3,4):
        img1 = img1[0]

    if img2.ndim == 3 and img2.shape[-1] not in (3,4):
        img2 = img2[0]

    diff = np.abs(
        img1.astype(np.float32)
        -
        img2.astype(np.float32)
    )

    plt.figure(figsize=(6,6))

    plt.imshow(diff,cmap="hot")

    plt.colorbar()

    plt.title(title)

    plt.axis("off")

    plt.savefig(
        MISMATCH_DIR /
        (filename + "_" + title + ".png"),
        dpi=200
    )

    plt.close()

In [28]:
if not np.array_equal(
    canonicalize_array(py["pixels"]),
    canonicalize_array(monai["pixels"])
):

    save_comparison_images(
        dcm_file.stem,
        py["pixels"],
        sitk_result["pixels"],
        monai["pixels"],
    )

    save_difference_image(
        dcm_file.stem,
        py["pixels"],
        monai["pixels"],
        "Py_vs_MONAI"
    )

In [29]:
def compare_metadata_values(py_ds, monai_meta):

    comparison = {}

    important = [

        "Rows",

        "Columns",

        "BitsAllocated",

        "BitsStored",

        "SamplesPerPixel",

        "PhotometricInterpretation",

        "PixelRepresentation",

        "NumberOfFrames",

    ]

    for tag in important:

        py_value = str(py_ds.get(tag,""))

        monai_value = str(
            monai_meta.get(tag,"")
        )

        comparison[tag] = py_value == monai_value

    return comparison

In [30]:
# row["Py_Throughput"] = throughput(
#     py["pixels"],
#     py["time"]
# )

# row["SimpleITK_Throughput"] = throughput(
#     sitk_result["pixels"],
#     sitk_result["time"]
# )

# row["MONAI_Throughput"] = throughput(
#     monai["pixels"],
#     monai["time"]
# )

In [31]:
def agreement_score(row):

    score = 0

    score += int(row["PyDICOM_OK"])
    score += int(row["SimpleITK_OK"])
    score += int(row["MONAI_OK"])

    score += int(row["Py_vs_MONAI_Equal"])
    score += int(row["Py_vs_SimpleITK_Equal"])
    score += int(row["SimpleITK_vs_MONAI_Equal"])

    return score

In [32]:
df["AgreementScore"] = df.apply(
    agreement_score,
    axis=1
)

In [33]:
print("=" * 80)
print("FINAL VALIDATION REPORT")
print("=" * 80)

print(f"Total DICOMs          : {len(df)}")
print(f"PyDICOM Success       : {df['PyDICOM_OK'].sum()}")
print(f"SimpleITK Success     : {df['SimpleITK_OK'].sum()}")
print(f"MONAI Success         : {df['MONAI_OK'].sum()}")

print()

print("Exact Pixel Agreement")

print(f"PyDICOM vs SimpleITK : {df['Py_vs_SimpleITK_Equal'].sum()}")
print(f"PyDICOM vs MONAI     : {df['Py_vs_MONAI_Equal'].sum()}")
print(f"SimpleITK vs MONAI   : {df['SimpleITK_vs_MONAI_Equal'].sum()}")

print()

print("Average Read Time (seconds)")

print(df[
    [
        "Py_Time",
        "SimpleITK_Time",
        "MONAI_Time"
    ]
].mean())

print()

print("Average Throughput (pixels/sec)")

print(df[
    [
        "Py_Throughput",
        "SimpleITK_Throughput",
        "MONAI_Throughput"
    ]
].mean())

FINAL VALIDATION REPORT
Total DICOMs          : 3468
PyDICOM Success       : 3468
SimpleITK Success     : 3468
MONAI Success         : 3468

Exact Pixel Agreement
PyDICOM vs SimpleITK : 2083
PyDICOM vs MONAI     : 0
SimpleITK vs MONAI   : 0

Average Read Time (seconds)
Py_Time           0.038962
SimpleITK_Time    0.095818
MONAI_Time        0.120159
dtype: float64

Average Throughput (pixels/sec)
Py_Throughput           5.245102e+08
SimpleITK_Throughput    2.004841e+08
MONAI_Throughput        1.827836e+08
dtype: float64


# VISUALIZATION (Plotting)

In [38]:
# ============================================================
# VISUALIZATION SETUP
# ============================================================

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

PLOT_DIR = OUTPUT_DIR / "Plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 150
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 11

In [39]:
def plot_reader_success(df):

    success = [
        df["PyDICOM_OK"].sum(),
        df["SimpleITK_OK"].sum(),
        df["MONAI_OK"].sum(),
    ]

    labels = [
        "PyDICOM",
        "SimpleITK",
        "MONAI",
    ]

    plt.figure(figsize=(6,4))

    plt.bar(labels, success)

    plt.ylabel("Successful Reads")

    plt.title("Reader Success")

    for i,v in enumerate(success):
        plt.text(i,v,f"{v}",ha="center",va="bottom")

    plt.tight_layout()

    plt.savefig(PLOT_DIR/"reader_success.png")

    plt.close()

In [40]:
def plot_runtime(df):

    plt.figure(figsize=(7,5))

    plt.boxplot([
        df["Py_Time"],
        df["SimpleITK_Time"],
        df["MONAI_Time"],
    ])

    plt.xticks(
        [1,2,3],
        [
            "PyDICOM",
            "SimpleITK",
            "MONAI",
        ]
    )

    plt.ylabel("Read Time (seconds)")

    plt.title("Reader Runtime Distribution")

    plt.grid(alpha=0.3)

    plt.tight_layout()

    plt.savefig(PLOT_DIR/"runtime_boxplot.png")

    plt.close()

In [41]:
def plot_transfer_syntax(df):

    counts = (
        df["TransferSyntax"]
        .value_counts()
        .sort_values()
    )

    plt.figure(figsize=(9,5))

    counts.plot(kind="barh")

    plt.xlabel("Number of Files")

    plt.title("Transfer Syntax Distribution")

    plt.tight_layout()

    plt.savefig(PLOT_DIR/"transfer_syntax.png")

    plt.close()

In [42]:
def plot_photometric(df):

    counts = (
        df["PhotometricInterpretation"]
        .value_counts()
        .sort_values()
    )

    plt.figure(figsize=(8,5))

    counts.plot(kind="barh")

    plt.xlabel("Number of Files")

    plt.title("Photometric Interpretation")

    plt.tight_layout()

    plt.savefig(PLOT_DIR/"photometric.png")

    plt.close()

In [43]:
def plot_manufacturer(df):

    counts = (
        df["Manufacturer"]
        .fillna("Unknown")
        .value_counts()
    )

    plt.figure(figsize=(9,5))

    counts.plot(kind="bar")

    plt.ylabel("Files")

    plt.title("Manufacturer Distribution")

    plt.xticks(rotation=45,ha="right")

    plt.tight_layout()

    plt.savefig(PLOT_DIR/"manufacturer.png")

    plt.close()

In [44]:
def plot_agreement_heatmap(df):

    matrix = np.array([
        [
            100,
            df["Py_vs_SimpleITK_Equal"].mean()*100,
            df["Py_vs_MONAI_Equal"].mean()*100,
        ],
        [
            df["Py_vs_SimpleITK_Equal"].mean()*100,
            100,
            df["SimpleITK_vs_MONAI_Equal"].mean()*100,
        ],
        [
            df["Py_vs_MONAI_Equal"].mean()*100,
            df["SimpleITK_vs_MONAI_Equal"].mean()*100,
            100,
        ]
    ])

    labels=[
        "PyDICOM",
        "SimpleITK",
        "MONAI",
    ]

    plt.figure(figsize=(6,5))

    plt.imshow(matrix)

    plt.xticks(range(3),labels)

    plt.yticks(range(3),labels)

    for i in range(3):
        for j in range(3):

            plt.text(
                j,
                i,
                f"{matrix[i,j]:.1f}%",
                ha="center",
                va="center",
                fontsize=11
            )

    plt.colorbar(label="Agreement (%)")

    plt.title("Reader Agreement")

    plt.tight_layout()

    plt.savefig(PLOT_DIR/"agreement_heatmap.png")

    plt.close()

In [45]:
def plot_rmse(df):

    plt.figure(figsize=(7,5))

    plt.hist(
        df["Py_vs_MONAI_RMSE"].dropna(),
        bins=40
    )

    plt.xlabel("RMSE")

    plt.ylabel("Number of Files")

    plt.title("PyDICOM vs MONAI RMSE")

    plt.tight_layout()

    plt.savefig(PLOT_DIR/"rmse_histogram.png")

    plt.close()

In [46]:
def plot_max_error(df):

    plt.figure(figsize=(7,5))

    plt.hist(
        df["Py_vs_MONAI_MaxError"].dropna(),
        bins=40
    )

    plt.xlabel("Maximum Pixel Error")

    plt.ylabel("Files")

    plt.title("Maximum Pixel Difference")

    plt.tight_layout()

    plt.savefig(PLOT_DIR/"max_error_histogram.png")

    plt.close()

In [47]:
def plot_throughput(df):

    mean_values=[

        df["Py_Throughput"].mean(),

        df["SimpleITK_Throughput"].mean(),

        df["MONAI_Throughput"].mean(),

    ]

    labels=[

        "PyDICOM",

        "SimpleITK",

        "MONAI",

    ]

    plt.figure(figsize=(6,4))

    plt.bar(labels,mean_values)

    plt.ylabel("Pixels / second")

    plt.title("Average Decoder Throughput")

    plt.tight_layout()

    plt.savefig(PLOT_DIR/"throughput.png")

    plt.close()

In [48]:
def plot_failure_reasons(df):

    counts=(
        df["FailureReason"]
        .fillna("PASS")
        .value_counts()
    )

    plt.figure(figsize=(7,7))

    plt.pie(
        counts.values,
        labels=counts.index,
        autopct="%1.1f%%"
    )

    plt.title("Failure Classification")

    plt.savefig(PLOT_DIR/"failure_reasons.png")

    plt.close()

In [49]:
def plot_failure_reasons(df):

    counts=(
        df["FailureReason"]
        .fillna("PASS")
        .value_counts()
    )

    plt.figure(figsize=(7,7))

    plt.pie(
        counts.values,
        labels=counts.index,
        autopct="%1.1f%%"
    )

    plt.title("Failure Classification")

    plt.savefig(PLOT_DIR/"failure_reasons.png")

    plt.close()

In [50]:
def plot_runtime_vs_size(df):

    pixels=[]

    for s in df["Py_Shape"]:

        try:

            shape=eval(s)

            pixels.append(np.prod(shape))

        except:

            pixels.append(np.nan)

    plt.figure(figsize=(7,5))

    plt.scatter(
        pixels,
        df["Py_Time"],
        s=12
    )

    plt.xlabel("Number of Pixels")

    plt.ylabel("Read Time (s)")

    plt.title("Runtime vs Image Size")

    plt.tight_layout()

    plt.savefig(PLOT_DIR/"runtime_vs_pixels.png")

    plt.close()

In [51]:
plot_reader_success(df)

plot_runtime(df)

plot_transfer_syntax(df)

plot_photometric(df)

plot_manufacturer(df)

plot_agreement_heatmap(df)

plot_rmse(df)

plot_max_error(df)

plot_throughput(df)

plot_failure_reasons(df)

plot_runtime_vs_size(df)

print(f"Plots saved to:\n{PLOT_DIR}")

Plots saved to:
benchmark_results\Plots
